In [1]:
import scipy.io
data = scipy.io.loadmat('E:/eeg-analysis/eeg-analysis/P01/S1/eeg/alldata_sbj01_sess1_RSraw.set')
print(data)

{'__header__': b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: Mon Jun 14 17:51:54 2021', '__version__': '1.0', '__globals__': [], 'setname': array([], dtype='<U1'), 'filename': array(['alldata_sbj01_sess1_RSraw.set'], dtype='<U29'), 'filepath': array(['D:\\m.hinss\\Desktop\\comeptition_done\\P01\\S1\\eeg'],
      dtype='<U46'), 'subject': array([], dtype='<U1'), 'group': array([], dtype='<U1'), 'condition': array([], dtype='<U1'), 'session': array([], shape=(0, 0), dtype=uint8), 'comments': array([], dtype='<U1'), 'nbchan': array([[62]], dtype=uint8), 'trials': array([[1]], dtype=uint8), 'pnts': array([[30000]], dtype=uint16), 'srate': array([[500]], dtype=uint16), 'xmin': array([[0]], dtype=uint8), 'xmax': array([[59.998]]), 'times': array([[    0,     2,     4, ..., 59994, 59996, 59998]],
      shape=(1, 30000), dtype=uint16), 'data': array([[ -9716.064  ,  -9728.76   ,  -9737.842  , ...,  -9609.131  ,
         -9602.539  ,  -9618.555  ],
       [  1573.9258 ,   1559.1797 ,   

In [7]:
# ============================================================
# CONVERT .MAT FILE TO CSV & FILTER TO 16 CHANNELS
# ============================================================
import pandas as pd
import numpy as np
import os

print("="*60)
print("CONVERTING .MAT FILE TO CSV")
print("="*60)

# Function to safely extract data from nested structures
def extract_nested_data(obj, max_depth=3, current_depth=0):
    """Recursively extract numeric arrays from nested MATLAB structures"""
    if current_depth > max_depth:
        return None
    
    if isinstance(obj, np.ndarray):
        if obj.dtype.names:  # Structured array
            for field_name in obj.dtype.names:
                field_data = obj[field_name][0, 0] if obj.shape == (1, 1) else obj[field_name]
                result = extract_nested_data(field_data, max_depth, current_depth + 1)
                if result is not None and result.size > 100:  # Assume EEG data has many samples
                    return result
        elif obj.size > 100 and len(obj.shape) >= 1:
            return obj
    return None

# Try to extract all possible data arrays
all_data_arrays = {}
print("Searching for data arrays...")

for key, value in data.items():
    if not key.startswith('__'):  # Skip MATLAB metadata
        extracted = extract_nested_data(value)
        if extracted is not None:
            all_data_arrays[key] = extracted

# Show found arrays
print(f"\nFound {len(all_data_arrays)} potential data arrays:")
for key, arr in all_data_arrays.items():
    print(f"- {key}: {arr.shape} {arr.dtype}")

# Convert each array to CSV
if all_data_arrays:
    for key, arr in all_data_arrays.items():
        try:
            print(f"\n🔄 Processing '{key}'...")
            
            # Prepare data for CSV
            if len(arr.shape) == 1:
                # 1D array - convert to column
                df = pd.DataFrame({f"{key}_data": arr})
            elif len(arr.shape) == 2:
                # 2D array - channels x time or time x channels
                if arr.shape[0] > arr.shape[1]:  # Likely time x channels
                    df = pd.DataFrame(arr, columns=[f"Channel_{i+1}" for i in range(arr.shape[1])])
                else:  # Likely channels x time, transpose
                    df = pd.DataFrame(arr.T, columns=[f"Channel_{i+1}" for i in range(arr.shape[0])])
            elif len(arr.shape) == 3:
                # 3D array - flatten to 2D (trials x channels x time -> (trials*time) x channels)
                reshaped = arr.reshape(-1, arr.shape[-1])
                df = pd.DataFrame(reshaped, columns=[f"Channel_{i+1}" for i in range(reshaped.shape[1])])
            else:
                # Higher dimensions - flatten to 2D
                reshaped = arr.reshape(arr.shape[0], -1)
                df = pd.DataFrame(reshaped, columns=[f"Feature_{i+1}" for i in range(reshaped.shape[1])])
            
            # Save to CSV (full dataset)
            output_file = f"E:/eeg-analysis/{key}_converted_data.csv"
            df.to_csv(output_file, index=False)
            
            print(f"✅ {key} saved to: {output_file}")
            print(f"   Shape: {df.shape}")
            print(f"   Size: {os.path.getsize(output_file) / (1024*1024):.2f} MB")
            print(f"   Preview:\n{df.head(3)}")
            
        except Exception as e:
            print(f"❌ Error converting {key}: {str(e)}")
else:
    print("\n❌ No data arrays found to convert!")
    print("Let's try a different approach...")
    
    # Alternative: try to convert all non-metadata keys directly
    for key, value in data.items():
        if not key.startswith('__'):
            try:
                if isinstance(value, np.ndarray) and value.size > 0:
                    # Force conversion to DataFrame
                    if len(value.shape) == 0:
                        continue  # Skip scalar values
                    elif len(value.shape) == 1:
                        df = pd.DataFrame({key: value.flatten()})
                    else:
                        df = pd.DataFrame(value.reshape(value.shape[0], -1))
                    
                    output_file = f"E:/eeg-analysis/{key}_raw_data.csv"
                    df.to_csv(output_file, index=False)
                    print(f"✅ Raw conversion: {key} -> {output_file}")
                    
            except Exception as e:
                print(f"❌ Raw conversion failed for {key}: {str(e)}")

print("\n" + "="*60)
print("CONVERSION SUMMARY")
print("="*60)
csv_files = [f for f in os.listdir("E:/eeg-analysis/") if f.endswith("_data.csv")]
if csv_files:
    print(f"Generated {len(csv_files)} CSV files:")
    for csv_file in csv_files:
        file_path = f"E:/eeg-analysis/{csv_file}"
        file_size = os.path.getsize(file_path) / (1024*1024)
        print(f"- {csv_file} ({file_size:.2f} MB)")
else:
    print("No CSV files were generated successfully.")

# ============================================================
# FILTER TO 16 SPECIFIC CHANNELS
# ============================================================
print("\n" + "="*60)
print("FILTERING TO 16 SELECTED CHANNELS")
print("="*60)

# Define the 16 channels you want to keep
selected_channels = [1, 2, 5, 6, 10, 11, 23, 24, 27, 28, 8, 9, 16, 17, 21, 22]
print(f"Selected channels: {sorted(selected_channels)}")

# Read the existing CSV file with all channels
input_csv = "E:/eeg-analysis/data_converted_data.csv"
print(f"\nLoading data from: {input_csv}")

try:
    # Load the full dataset
    df_full = pd.read_csv(input_csv)
    print(f"Original dataset shape: {df_full.shape}")
    print(f"Original columns: {list(df_full.columns)}")
    
    # Create column names for selected channels
    selected_column_names = [f"Channel_{i}" for i in selected_channels]
    
    # Check which channels actually exist in the data
    available_channels = []
    missing_channels = []
    
    for channel_name in selected_column_names:
        if channel_name in df_full.columns:
            available_channels.append(channel_name)
        else:
            missing_channels.append(channel_name)
    
    print(f"\nAvailable channels: {len(available_channels)}")
    print(f"Available: {available_channels}")
    
    if missing_channels:
        print(f"\nMissing channels: {missing_channels}")
    
    if available_channels:
        # Filter the dataset to include only selected channels
        df_filtered = df_full[available_channels].copy()
        
        print(f"\nFiltered dataset shape: {df_filtered.shape}")
        print(f"Filtered columns: {list(df_filtered.columns)}")
        
        # Save the filtered dataset
        output_file_filtered = "E:/eeg-analysis/data_16channels_filtered.csv"
        df_filtered.to_csv(output_file_filtered, index=False)
        
        print(f"\n✅ Filtered dataset saved to: {output_file_filtered}")
        print(f"✅ File size: {os.path.getsize(output_file_filtered) / (1024*1024):.2f} MB")
        
        # Show preview of filtered data
        print(f"\nPreview of filtered data:")
        print(df_filtered.head())
        
        print(f"\nDataset statistics:")
        print(f"- Number of time samples: {df_filtered.shape[0]:,}")
        print(f"- Number of channels: {df_filtered.shape[1]}")
        print(f"- Data types: {df_filtered.dtypes.unique()}")
        
    else:
        print("❌ None of the selected channels were found in the dataset!")
        
except FileNotFoundError:
    print(f"❌ Error: Could not find the input file: {input_csv}")
    print("Please make sure the conversion was successful!")
    
except Exception as e:
    print(f"❌ Error filtering channels: {str(e)}")

print("\n" + "="*60)
print("COMPLETE: CONVERSION & FILTERING DONE!")
print("="*60)

CONVERTING .MAT FILE TO CSV
Searching for data arrays...

Found 3 potential data arrays:
- times: (1, 30000) uint16
- data: (62, 30000) float32
- etc: (1, 853) object

🔄 Processing 'times'...
✅ times saved to: E:/eeg-analysis/times_converted_data.csv
   Shape: (30000, 1)
   Size: 0.19 MB
   Preview:
   Channel_1
0          0
1          2
2          4

🔄 Processing 'data'...
✅ data saved to: E:/eeg-analysis/data_converted_data.csv
   Shape: (30000, 62)
   Size: 18.49 MB
   Preview:
     Channel_1    Channel_2    Channel_3   Channel_4    Channel_5  \
0 -9716.064453  1573.925781 -4663.525391 -923.242188 -7666.259766   
1 -9728.759766  1559.179688 -4677.001953 -937.646484 -7680.664062   
2 -9737.841797  1550.927734 -4685.498047 -947.900391 -7689.062500   

      Channel_6    Channel_7    Channel_8    Channel_9  Channel_10  ...  \
0  10242.285156 -1076.611328 -2649.414062 -3239.062500  707.763672  ...   
1  10227.978516 -1089.697266 -2665.087891 -3252.490234  695.410156  ...   
2  10218.505